In [ ]:
import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
from datasets import DatasetDict
from transformers import pipeline
from transformers import DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainingArguments
from transformers import Seq2SeqTrainer
import evaluate


In [61]:
raw_datasets= load_dataset("ngia/translation-en-fr")

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/30 [00:00<?, ?it/s]

In [62]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['english_src', 'french_tgt'],
        num_rows: 40252491
    })
    test: Dataset({
        features: ['english_src', 'french_tgt'],
        num_rows: 406591
    })
})

In [63]:
raw_datasets['train'][0]

{'english_src': 'Mr President, the world gets smaller every day, and this is now so obviously the case that the globalization of the economy is something which everyone accepts as normal.',
 'french_tgt': "Monsieur le Président, le monde devient de plus en plus petit, ce constat est d'autant plus évident que la globalisation ou mondialisation de l'économie est un phénomène que tout le monde trouve naturel."}

In [64]:
small_datasets = DatasetDict({
    "train": raw_datasets["train"].shuffle(seed=42).select(range(10000)),
    "test": raw_datasets["test"].shuffle(seed=42).select(range(2000)),
})
small_datasets

DatasetDict({
    train: Dataset({
        features: ['english_src', 'french_tgt'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['english_src', 'french_tgt'],
        num_rows: 2000
    })
})

In [65]:
model_checkpoint = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, return_tensors="pt")
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [66]:
def preprocess_function(examples):
    inputs = examples["english_src"]
    targets = examples["french_tgt"]

    model_inputs = tokenizer(
        inputs,
        text_target=targets,
        max_length=128,
        truncation=True,
    )
    return model_inputs

tokenized_datasets= small_datasets.map(preprocess_function, batched=True, remove_columns=small_datasets["train"].column_names)
tokenized_datasets

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

OSError: [Errno 28] No space left on device

In [46]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [47]:
batch = data_collator([tokenized_datasets["train"][i] for i in range(1, 3)])
batch.keys()

KeysView({'input_ids': tensor([[ 1438,  4871,    37,  8412, 23803,  1437,   365,    37, 13598,  2786,
         48224,    48,    77,  6087,   160,   292,    32,   494,    30,     4,
         24930,  1736,   106,  2115,     3,     0, 59513, 59513, 59513, 59513,
         59513],
        [  660,  3950,    52,    73, 10520,   761,    46, 19654,    48,   218,
            52,  4465,   420,  1942,    21,  1876,  1130,    24, 42343,  1105,
           232,   544,    21, 11267,     9,     6, 19654,    71,   967,   102,
             0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1]]), 'labels': tensor([[ 9593,    22,   230,    37,  8412,  6485, 20596,    78, 13598,  2786,
          2916,  1349,    43,    38,  1780,   335,   230,    43,  2200,    36,
            19,   414,  1232,    13, 33584, 42505,   10

In [58]:
metric = evaluate.load("sacrebleu")

In [60]:
args= Seq2SeqTrainingArguments(
    "test-translation",
    # evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    # save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    push_to_hub=False,
)

trainer= Seq2SeqTrainer(
    model, 
    args, 
    train_dataset=tokenized_datasets["train"], 
    eval_dataset=tokenized_datasets["test"], 
    data_collator=data_collator, 
    # tokenizer= tokenizer, 
    compute_metrics=metric.compute
)
trainer.train()

Step,Training Loss
500,1.385740
1000,1.320178
1500,1.312660
2000,1.337928
2500,1.332231
3000,1.310166
3500,1.319343
4000,1.315316
4500,1.331114
5000,1.300939


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SafetensorError: Error while serializing: I/O error: No space left on device (os error 28)